# Namespaces – Development und Production

Zwei gleichnamige Webserver in getrennten Namespaces. Das Notebook aus diesem Verzeichnis mit dem Python-Kernel ausführen. Die Namen development und production müssen für diese Übung frei sein; keine vorhandenen fremden Namespaces verwenden oder löschen.

## 1. Vorhandene Namespaces
Namespaces gruppieren Ressourcen logisch. Nodes sind dagegen clusterweit.

In [ ]:
%%bash
set -e
kubectl get namespaces
kubectl get pods -n kube-system
kubectl get pods -A
kubectl get nodes

## 2. Namespaces erzeugen
Development imperativ, Production deklarativ. Die folgenden Befehle nur ausführen, wenn beide Namespaces noch nicht existieren.

In [ ]:
%%bash
set -e
if kubectl get namespace development >/dev/null 2>&1 || kubectl get namespace production >/dev/null 2>&1; then echo 'Namespaces bereits vorhanden – vor Fortsetzung prüfen.'; exit 1; fi
kubectl create namespace development
cat production-ns.yml
kubectl apply -f production-ns.yml

## 3. Gleichnamige Pods bereitstellen
Die Namespace-Zuordnung erfolgt über metadata.namespace. Development enthält zusätzlich eine Toolbox mit curl für den späteren Netzwerktest. Die Installation von curl benötigt Internetzugriff.

In [ ]:
%%bash
set -e
cat development.yml production.yml
kubectl apply -f development.yml
kubectl apply -f production.yml
kubectl wait -n development --for=condition=Ready pod/webserver --timeout=180s
kubectl wait -n production --for=condition=Ready pod/webserver --timeout=180s

## 4. Ressourcen anzeigen
Der Name webserver ist innerhalb jedes Namespace eindeutig. Ohne Namespace-Option wird der Namespace des aktuellen Kontexts verwendet (standardmäßig default).

In [ ]:
%%bash
set -e
kubectl get pods
kubectl get pods -n development
kubectl get pods -n production
kubectl get pods -A

Der folgende Aufruf soll fehlschlagen, sofern kein gleichnamiger Pod im Namespace default existiert.

In [ ]:
%%bash
set -e
kubectl describe pod webserver -n default || echo 'Erwartet: Pod im Namespace default nicht gefunden.'

## 5. Anwendungen aufrufen
In zwei separaten VS-Code-Terminals ausführen:

```bash
kubectl port-forward -n development pod/webserver 8081:80
```

```bash
kubectl port-forward -n production pod/webserver 8082:80
```

Im Browser http://localhost:8081 und http://localhost:8082 öffnen. Development zeigt Version 2.0, Production Version 1.0. Anschließend beide Port-Forward-Prozesse mit Strg+C beenden.

## 6. Standard-Namespace wechseln
Die Einstellung betrifft den aktuellen kubectl-Kontext und bleibt über einzelne Befehle hinaus bestehen. Der ursprüngliche Namespace wird am Ende dieser Zelle wiederhergestellt.

In [ ]:
%%bash
set -e
original=$(kubectl config view --minify -o jsonpath='{..namespace}')
trap 'kubectl config set-context --current --namespace="${original:-default}"' EXIT
kubectl config set-context --current --namespace=development
kubectl get pods
kubectl config set-context --current --namespace=production
kubectl get pods

## 7. Netzwerk-Isolation
Namespaces allein isolieren das Netzwerk nicht. Ohne einschränkende NetworkPolicy kann die Toolbox im Development-Pod die Pod-IP von Production direkt erreichen.

In [ ]:
%%bash
set -e
PROD_IP=$(kubectl get pod webserver -n production -o jsonpath='{.status.podIP}')
echo "$PROD_IP"
kubectl exec -n development webserver -c toolbox -- curl -fsS --max-time 10 "http://$PROD_IP/"

## 8. Namespace löschen
Das Löschen eines Namespace entfernt auch seine enthaltenen Ressourcen. Nur die in dieser Übung selbst angelegten Namespaces löschen.

In [ ]:
%%bash
set -e
kubectl delete namespace development
kubectl get pods -n production
kubectl get namespaces

In [ ]:
%%bash
set -e
kubectl delete namespace production
kubectl get namespaces

## Zusammenfassung
Namespaces ermöglichen gleichnamige Ressourcen, logische Gruppierung und einen gemeinsamen Lebenszyklus. Sie sind keine automatische Netzwerk- oder Sicherheitsgrenze.